In [1]:
import sys
import os

# This makes imports work from the project root
project_root = os.path.abspath('..')   # go up one level from notebook
sys.path.insert(0, project_root)

from include.helper_functions import *

In [2]:
#read all csv
import os
import pandas as pd

# Define the path to the data directory
data_dir = "../data/raw/csv/ttc_subway_delay"

# List all CSV files in the data directory
csv_files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]

years_list = ['2023', '2024', '2025']
# Read each CSV file into a DataFrame based on year in filename and store in a list
dfs = []
for file in csv_files:
    # Extract year from filename (assuming format like "fafd_afdaf_2020.csv")   
    year = file.split('.')[0][-4:]  # Get the last 4 characters (the year)
    # Only process files for years in the specified list
    if str(year) in years_list:
        df = pd.read_csv(os.path.join(data_dir, file))
        df['year'] = year  # Add a column for the year
        dfs.append(df)
        print(f"Loaded {file}")

# Concatenate all DataFrames into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)
print("All CSV files have been combined into a single DataFrame.")

#drop id column
if 'id' in combined_df.columns:
    combined_df = combined_df.drop(columns=['id'])
    print("Dropped 'id' column from the combined DataFrame.")

#delete duplicate rows and print count of duplicate rows
duplicate_rows = combined_df[combined_df.duplicated(keep=False)]
print("Count of duplicate rows:", len(duplicate_rows))
combined_df = combined_df.drop_duplicates()
print("Removed duplicate rows from the combined DataFrame.")

#rename all the columns in combined_df as per dictionary
column_mapping = {
    'Date': 'date',
    'Time': 'time',
    'Day': 'day',
    'Station': 'station',
    'Code': 'code',
    'Min Delay': 'delayed_minutes',
    'Min Gap': 'gap_minutes',
    'Bound': 'bound',
    'Line': 'line',
    'Vehicle': 'vehicle'
}

# Rename columns in combined_df
combined_df = combined_df.rename(columns=column_mapping)

# Convert the 'date' column from string to a proper datetime object
# errors='coerce' turns any unparseable values into NaT instead of crashing
combined_df['date'] = pd.to_datetime(combined_df['date'], errors='coerce')

# Combine the 'date' and 'time' columns into a single datetime column
# This gives us a precise timestamp for each delay event (e.g. 2024-01-01 08:35:00)
combined_df['datetime'] = pd.to_datetime(combined_df['date'].astype(str) + ' ' + combined_df['time'], errors='coerce')

# Display the first few rows of the combined DataFrame
print(combined_df.head())

# Extract hour of day (0–23) — captures patterns like rush hour
combined_df['hour'] = combined_df['datetime'].dt.hour

# Extract day of week as a number (0=Monday, 6=Sunday)
# Useful because weekdays typically have different delay patterns than weekends
combined_df['weekday'] = combined_df['date'].dt.weekday

# Binary flag: 1 if Saturday or Sunday, 0 otherwise
# Cast to int so it's stored as 0/1 instead of True/False
combined_df['is_weekend'] = (combined_df['weekday'] >= 5).astype(int)
# Extract month (1–12) — captures seasonal patterns (e.g. winter delays)
combined_df['month'] = combined_df['date'].dt.month

# Extract ISO week number (1–52) — finer-grained seasonal signal than month
combined_df['week'] = combined_df['date'].dt.isocalendar().week.astype(int)

# Peak hour flag: 1 during AM rush (7–9) or PM rush (16–19), 0 otherwise
# These windows typically see the most congestion on TTC
combined_df['peak_hour'] = combined_df['hour'].apply(lambda x: 1 if (7 <= x <= 9) or (16 <= x <= 19) else 0)

#write in data/processed/combined_data.csv# Define the path to the output file
output_file = "../data/processed/ttc_subway_delay_data_combined.csv"
# Save the combined DataFrame to a new CSV file
combined_df.to_csv(output_file, index=False)
print(f"Combined data has been saved to {output_file}.")

Loaded ttc-subway-delay-data-2024.csv
Loaded ttc-subway-delay-data-2025.csv
Loaded ttc-subway-delay-data-2023.csv
All CSV files have been combined into a single DataFrame.
Dropped 'id' column from the combined DataFrame.
Count of duplicate rows: 160
Removed duplicate rows from the combined DataFrame.
        date   time     day             station   code  delayed_minutes  \
0 2024-01-01  02:00  Monday    SHEPPARD STATION    MUI                0   
1 2024-01-01  02:00  Monday      DUNDAS STATION   MUIS                0   
2 2024-01-01  02:08  Monday      DUNDAS STATION  MUPAA                4   
3 2024-01-01  02:13  Monday  KENNEDY BD STATION  PUTDN               10   
4 2024-01-01  02:22  Monday       BLOOR STATION  MUPAA                4   

   gap_minutes bound line  vehicle  year            datetime  
0            0     N   YU     5491  2024 2024-01-01 02:00:00  
1            0     N   YU        0  2024 2024-01-01 02:00:00  
2           10     N   YU     6051  2024 2024-01-01 02:08:

In [3]:
#read ttc station mapping file in processed folder
mapping_ttc_station_file = "../data/processed/mapping_ttc_station.csv"
mapping_ttc_station_df = pd.read_csv(mapping_ttc_station_file)

#read ttc delay code mapping file in processed folder
mapping_ttc_delay_code_file = "../data/processed/mapping_ttc_delay_code.csv"
mapping_ttc_delay_code_df = pd.read_csv(mapping_ttc_delay_code_file)

#read ttc subway station master file code mapping file in processed folder
mapping_ttc_subway_station_master_file = "../data/master/ttc_subway_station_master.csv"
mapping_ttc_subway_station_master_df = pd.read_csv(mapping_ttc_subway_station_master_file)


#print count of duplicate rows
duplicate_rows_mapping = mapping_ttc_station_df[mapping_ttc_station_df.duplicated(keep=False)]
print("Count of duplicate rows in the mapping DataFrame:", len(duplicate_rows_mapping))

#find duplicate station names in the mapping file
duplicate_station_names = mapping_ttc_station_df[mapping_ttc_station_df.duplicated(subset=['station'], keep=False)]
print("Duplicate station names in the mapping DataFrame:")
print(duplicate_station_names)


#print count of duplicate rows
duplicate_rows_mapping = mapping_ttc_delay_code_df[mapping_ttc_delay_code_df.duplicated(keep=False)]
print("Count of duplicate rows in the mapping DataFrame:", len(duplicate_rows_mapping))

#find duplicate station names in the mapping file
duplicate_station_names = mapping_ttc_delay_code_df[mapping_ttc_delay_code_df.duplicated(subset=['delay_code'], keep=False)]
print("Duplicate station names in the mapping DataFrame:")
print(duplicate_station_names)

#print count of duplicate rows
duplicate_rows_mapping = mapping_ttc_subway_station_master_df[mapping_ttc_subway_station_master_df.duplicated(keep=False)]
print("Count of duplicate rows in the mapping DataFrame:", len(duplicate_rows_mapping))

#find duplicate station names in the mapping file
duplicate_station_names = mapping_ttc_subway_station_master_df[mapping_ttc_subway_station_master_df.duplicated(subset=['station'], keep=False)]
print("Duplicate station names in the mapping DataFrame:")
print(duplicate_station_names)



Count of duplicate rows in the mapping DataFrame: 0
Duplicate station names in the mapping DataFrame:
Empty DataFrame
Columns: [station, station_clean, row_count, mapped_station, score, include_station, remarks]
Index: []
Count of duplicate rows in the mapping DataFrame: 0
Duplicate station names in the mapping DataFrame:
Empty DataFrame
Columns: [delay_code, row_count, mapped_delay_code, fuzzy_score, include_code, remarks]
Index: []
Count of duplicate rows in the mapping DataFrame: 0
Duplicate station names in the mapping DataFrame:
Empty DataFrame
Columns: [station, line, location, grade]
Index: []


In [4]:
# Merge combined_df with mapping__ttc_station_df on 'station' column and rename columns
merged_df = pd.merge(combined_df, mapping_ttc_station_df, on='station', how='left') \
    .rename(columns={'score': 'station_score', 'remarks': 'station_remarks'}) \
    .drop(columns=['row_count', 'station_clean'])

# Merge merged_df with mapping__ttc_delay_code_df on 'code' column
merged_df = pd.merge(merged_df, mapping_ttc_delay_code_df, left_on='code', right_on='delay_code', how='left') \
    .rename(columns={'score': 'delay_code_score', 'remarks': 'delay_code_remarks', 'description': 'delay_code_description'}) \
    .drop(columns=['row_count'])

#rename the column from right side before merging with subway station master file
mapping_ttc_subway_station_master_df = mapping_ttc_subway_station_master_df.rename(columns={'station': 'subway_station', 'line': 'line_clean'})
merged_df = pd.merge(merged_df, mapping_ttc_subway_station_master_df, left_on='mapped_station', right_on='subway_station', how='left') \
    .drop(columns=['subway_station','location','grade'])
print("Merged DataFrame has been created by combining the combined DataFrame with the mapping DataFrames.")
#print the columns of merged_df
print("Columns in the merged DataFrame:")
print(merged_df.columns.tolist())


Merged DataFrame has been created by combining the combined DataFrame with the mapping DataFrames.
Columns in the merged DataFrame:
['date', 'time', 'day', 'station', 'code', 'delayed_minutes', 'gap_minutes', 'bound', 'line', 'vehicle', 'year', 'datetime', 'hour', 'weekday', 'is_weekend', 'month', 'week', 'peak_hour', 'mapped_station', 'station_score', 'include_station', 'station_remarks', 'delay_code', 'mapped_delay_code', 'fuzzy_score', 'include_code', 'delay_code_remarks', 'line_clean']


In [5]:
#clean data for bound column using canonical_bound function and create a new column bound_clean
merged_df["bound_clean"] = merged_df["bound"].apply(canonical_bound)

#use imputer to fill missing values in bound_clean column based on line and station
from sklearn.impute import SimpleImputer

# With fallback to global most frequent
global_mode = merged_df['bound_clean'].mode()[0]

merged_df['bound_clean'] = merged_df.groupby(['line_clean', 'mapped_station'])['bound_clean'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else global_mode)
)



In [6]:
# Identify outliers in 'Min Delay' and 'Min Gap' using MAD method within each line
# remove the rows with negative delay and 0 delay
merged_df = merged_df[merged_df["delayed_minutes"] > 0]

Z_THRESH = 3.5  # Adjust this threshold as needed
    

out_delay = merged_df["delayed_minutes"].transform(lambda s: mad_outlier_mask(s, z_thresh=Z_THRESH))

mask_outlier = out_delay 
print("\nRows flagged as outliers:", int(mask_outlier.sum()))

merged_df_clean = merged_df.loc[~mask_outlier].copy()
print("Final cleaned dataset shape:", merged_df_clean.shape)


Rows flagged as outliers: 2329
Final cleaned dataset shape: (24389, 29)


In [7]:

#write merged_df to obt csv file in processed folder
output_file_merged = "../data/processed/ttc_subway_delay_data_obt.csv"
merged_df_clean.to_csv(output_file_merged, index=False)
print(f"Merged data has been saved to {output_file_merged}.")

Merged data has been saved to ../data/processed/ttc_subway_delay_data_obt.csv.


In [8]:
# read data from ttc_subway_delay_data_combined.csv into data frame and display the first few rows
import pandas as pd

# Define the path to the input file
input_file = "../data/processed/ttc_subway_delay_data_obt.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(input_file)

# Display the first few rows of the DataFrame
print(df.head())


         date   time     day             station   code  delayed_minutes  \
0  2024-01-01  02:08  Monday      DUNDAS STATION  MUPAA                4   
1  2024-01-01  02:13  Monday  KENNEDY BD STATION  PUTDN               10   
2  2024-01-01  02:22  Monday       BLOOR STATION  MUPAA                4   
3  2024-01-01  02:25  Monday    ST CLAIR STATION  MUPAA                3   
4  2024-01-01  02:27  Monday    WOODBINE STATION   EUDO                7   

   gap_minutes bound line  vehicle  ...  station_score include_station  \
0           10     N   YU     6051  ...             90             1.0   
1           16     E   BD     5284  ...             95             1.0   
2           10     N   YU     5986  ...             90             1.0   
3            9     N   YU     6051  ...             95             1.0   
4           13     E   BD     5077  ...            100             1.0   

            station_remarks  delay_code  mapped_delay_code  fuzzy_score  \
0                      

In [9]:
# #print distince lines in the data frame
# distinct_lines = df['line_clean'].value_counts()
# print(distinct_lines)

In [10]:
# #get distinct codes with count from the data frame
# distinct_codes = df['mapped_delay_code'].value_counts()
# print(distinct_codes)

In [11]:
#read all the weather data csv files in the data/raw/csv/weather folder and concatenate them into a single data frame
import os
import pandas as pd

weather_data_folder = "../data/raw/csv/toronto_weather_data/"
weather_files = [f for f in os.listdir(weather_data_folder) if f.endswith(".csv")]

years_list = ['2023', '2024', '2025']
weather_dfs = []
for file in weather_files:
    year = file.split('.')[0][0:4]  # Get the last 4 characters (the year)
    if str(year) in years_list:
        file_path = os.path.join(weather_data_folder, file)
        df = pd.read_csv(file_path)
        weather_dfs.append(df)
        print(f"Loaded {file}")

combined_weather_df = pd.concat(weather_dfs, ignore_index=True)
print(combined_weather_df.head())

Loaded 2023_6.csv
Loaded 2023_7.csv
Loaded 2025_1.csv
Loaded 2025_3.csv
Loaded 2023_5.csv
Loaded 2023_4.csv
Loaded 2025_2.csv
Loaded 2023_12.csv
Loaded 2025_6.csv
Loaded 2023_1.csv
Loaded 2025_7.csv
Loaded 2023_11.csv
Loaded 2025_5.csv
Loaded 2023_3.csv
Loaded 2023_2.csv
Loaded 2025_4.csv
Loaded 2023_10.csv
Loaded 2024_3.csv
Loaded 2024_2.csv
Loaded 2024_1.csv
Loaded 2024_5.csv
Loaded 2025_10.csv
Loaded 2025_11.csv
Loaded 2024_4.csv
Loaded 2024_6.csv
Loaded 2025_12.csv
Loaded 2024_7.csv
Loaded 2024_9.csv
Loaded 2024_8.csv
Loaded 2025_9.csv
Loaded 2025_8.csv
Loaded 2024_12.csv
Loaded 2023_9.csv
Loaded 2023_8.csv
Loaded 2024_11.csv
Loaded 2024_10.csv
   Longitude (x)  Latitude (y)         Station Name  Climate ID  \
0          -79.4         43.63  TORONTO CITY CENTRE     6158359   
1          -79.4         43.63  TORONTO CITY CENTRE     6158359   
2          -79.4         43.63  TORONTO CITY CENTRE     6158359   
3          -79.4         43.63  TORONTO CITY CENTRE     6158359   
4       

In [36]:
print(combined_weather_df.shape)

(26304, 31)


In [19]:
combined_weather_df['Date/Time (LST)'] = pd.to_datetime(combined_weather_df['Date/Time (LST)'], errors='coerce')
combined_weather_df['Date/Time (LST)'] = combined_weather_df['Date/Time (LST)'].dt.floor('h')
#truncate the datetime to hour level
merged_df_clean['datetime'] = merged_df_clean['datetime'].dt.floor('h')
# Merge weather data with the combined subway delay data
merged_df = pd.merge(merged_df_clean, combined_weather_df, left_on=['datetime'], right_on=['Date/Time (LST)'], how='left')

#write output_file_merged_with_weather to obt csv file in processed folder
output_file_merged_with_weather = "../data/processed/ttc_subway_delay_data_weather_obt.csv"
merged_df.to_csv(output_file_merged_with_weather, index=False)
print(f"Merged data with weather has been saved to {output_file_merged_with_weather}.")

Merged data with weather has been saved to ../data/processed/ttc_subway_delay_data_weather_obt.csv.


In [22]:
# print min, max, mean, median, std of all column in merged_df

all_columns_stats = merged_df.describe()
print(all_columns_stats)

                             date  delayed_minutes   gap_minutes  \
count                       24389     24389.000000  24389.000000   
mean   2024-07-12 09:12:59.810570         5.550740      9.303046   
min           2023-01-01 00:00:00         2.000000      0.000000   
25%           2023-10-26 00:00:00         3.000000      7.000000   
50%           2024-07-21 00:00:00         5.000000      9.000000   
75%           2025-03-29 00:00:00         7.000000     11.000000   
max           2025-12-31 00:00:00        15.000000     30.000000   
std                           NaN         2.862811      3.796764   

            vehicle                    datetime          hour       weekday  \
count  24389.000000                       24389  24389.000000  24389.000000   
mean    5427.944524  2024-07-12 22:10:38.992988     12.960884      2.797122   
min        0.000000         2023-01-01 02:00:00      0.000000      0.000000   
25%     5216.000000         2023-10-26 15:00:00      8.000000      1.00